# Day 9: AI Agents + Text-to-SQL

#AI Agents VS Chatbots
Tool calling (Multi step reasoning)
Text-to-SQL pipeline-schema Injection,
Groq + SQLite


## AI Agent

AI Agent is an LLM that can:
- User tools
- Take multiple steps
- Decide which to take
- Act on the real world

Unlike chatbots: that only answer from memory, agents can search, query databases, run code, and call APIs.

Components:
- LLM -Brain/Decision Maker
- Tools - Function it can call
- Memory - Conversation history
- Reasoning - Multi-step planning

LLM + SQLite tools +

#Tool Calling
- giving the AI the ability to run specific Python functions. You define the tools, The AI decides when to call them
- get_schema() - return database table structure to the AI so it knows column names and types
- generate_sql(question) - search user question + schema to Groq

# Multi-Step Reasoning: The ReAct Pattern
ReAct = Reasoning +Acting. The agent loops through four steps:

Think - Understand the user's intent. What do you want?

Plan - What SQL is related? Which columns, filters, sort order?

Act - Execute the SQL on the real database.Fetch actual data

Respond - Format results

# Text-to-SQL
Step 1: Schema Injection
Tell the LLM what the database load like - table name, column name, data types, sample rows

Step 2: SQL Generation
LLM reads schema + user question and when a valid SQL query temperature = 0.0

Step 3: Execute + Respond

# The complete Text-to-SQL Pipeline
Load csv into SQLite

Get Database Schema

Generate SQL with Groq

Execute SQL on SQLite

Natural Language Answer

#Multi-Step Reasoning
user="prompt"
Step 1 - Think:".." Answer :".."



In [31]:
!pip install groq -q                                 # install groq
print("Libraries installed successfully")

Libraries installed successfully


In [32]:
import pandas as pd
import sqlite3
import io
import re
import os
from groq import Groq
print("All necessary  packages installed successfully ")

All necessary  packages installed successfully 


In [ ]:
import os
os.environ["GROQ_API_KEY"]="YOUR_GROQ_API"
client = Groq(api_key=os.environ["GROQ_API_KEY"])
model ="llama-3.1-8b-instant"
print("Groq client installed successfully")
print(f"Using model:{model}")


Groq client installed successfully
Using model:llama-3.1-8b-instant


In [34]:
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df =pd.read_csv(io.StringIO(csv_data))

# df.to_sql("students",conn,if_exists="replace",index=False)
print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print("\n First 5 rows:")
df.head()

Dataset loaded: 30 rows, 8 columns

 First 5 rows:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [35]:
conn = sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("Database created: college.db")
print("Table 'students' created with 30 student records")
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f"\nVerification: {test_df['total_rows'][0]} rows in database")

Database created: college.db
Table 'students' created with 30 student records

Verification: 30 rows in database


In [36]:
def get_schema(conn, table_name ="students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns:")
  for col in columns:
    schema_lines.append(f"-{col[1]} ({col[2]})")
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows=cursor.fetchall()
  schema_lines.append("\n Sample rows {first 3}")
  for row in sample_rows:
    schema_lines.append(f"{row}")
  return "\n".join(schema_lines)
schema = get_schema(conn)
print(schema)

Table: students
Columns:
-student_id (INTEGER)
-name (TEXT)
-age (INTEGER)
-gender (TEXT)
-subject (TEXT)
-marks (INTEGER)
-attendance (INTEGER)
-grade (TEXT)

 Sample rows {first 3}
(1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
(2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
(3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [37]:
def generate_sql(user_question, schema_text, client, mode):
  system_prompt=f""" You are an expert SQL assistant.
  You are connected to a SQLite database with the following structure:
  {schema_text}
  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or text -only the SQL query.
  3. Do not use markdown code block. Return the raw SQL only.
  4. The table name is: students
  5. Only use column names that exists in the schema above.
  6. Use single quotes for string values in WHERE clauses (example:WHERE subject ="Programming")
  7.If the user asks for top N,use ORDER BY marks DESC LIMIT N
  """
  response = client.chat.completions.create(model=model,
  messages=[
         {"role":"system","content":system_prompt},
         {"role": "user","content":user_question   }
         ],
  temperature=0.0)
  sql_query=response.choices[0].message.content.strip()
  return sql_query
question="Show me all female students"
print(f"Question:{question}")
print(f"\nGenerating SQL...")

sql = generate_sql(question, schema, client, model)
print(f"\nGenerated SQL:\n{sql}")

Question:Show me all female students

Generating SQL...

Generated SQL:
SELECT * FROM students WHERE gender = 'Female'


In [38]:
def execute_sql(sql_query,conn):
  """ cleans the AI-generated SQL and executes it on the SQLite database.
  Returns the results as a pandas DataFrame."""
  clean_sql = sql_query.strip()
  clean_sql = re.sub(r'```sql\s*', '', clean_sql)
  try:
    result_df=pd.read_sql_query(clean_sql,conn)
    return result_df, None
  except Exception as e:
    return None, str(e)


In [39]:
result, error = execute_sql(sql, conn)

if error:
    print(f"\nError: {error}")
else:
    print(f"\nQuery returned {len(result)} rows")
    print(result)


Query returned 15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female  Mathematics     58          73   
11          24       Usha Iy

In [40]:
def text_to_sql_agent(user_question, conn, client, model, verbose=True):
    print("=" * 60)
    print(f"User Question: {user_question}")
    print("=" * 60)
    if verbose:
        print("\n[STEP 1] Reading database schema...")
    schema_text = get_schema(conn)
    if verbose:
        print("Schema loaded successfully")
    if verbose:
        print("\n[STEP 2] Generating SQL query with Groq LLM...")
    generated_sql = generate_sql(user_question, schema_text, client, model)
    if verbose:
        print(f"Generated SQL:\n{generated_sql}")
    if verbose:
        print("\n[STEP 3] Executing SQL on the database...")
    result_df, error = execute_sql(generated_sql, conn)
    if error:
        print(f"SQL Execution Error: {error}")
        return None, generated_sql
    if verbose:
        print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
        print("\nRESULTS:")
        print("-" * 40)
        print(result_df.to_string(index=False))
        print("=" * 60)
    return result_df, generated_sql
result, sql_used = text_to_sql_agent(
    "Show top 5 students in Programming",
    conn,
    client,
    model
)

User Question: Show top 5 students in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+


In [41]:
result1,_= text_to_sql_agent("Show me all students who study Mathematics",conn,client,model)
result2,_ = text_to_sql_agent("What is the average marks for each subject",conn,client,model)
result3, _ = text_to_sql_agent("Show students who scored more than 90 marks",conn,client,model)
result4,_ = text_to_sql_agent("How many male and female students are there",conn,client,model)
result5,_ = text_to_sql_agent("Show female students who scored above 85 in Science or Programming , ordered by marks",conn,client,model)

User Question: Show me all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Fe

In [42]:
def generate_answer(user_question, query_results_df, client, model):
  if query_results_df is None or len(query_results_df)==0:
    return "No results were found for your query"
  results_text = query_results_df.to_string(index=False)
  prompt = f"""The user asked: '{user_question}'
  The database returned these results:
  {results_text}
  Please write a clear, friendly, 2-3 sentences answer to the user's question based on these results
  Be specific, Mention actual names and numbers from the data.
  Do not add information not present in the results."""

  response = client.chat.completions.create(
       model=model,
       messages =[{"role":"user","content": prompt}],
       temperature = 0.3
   )
  return response.choices[0].message.content.strip()

In [43]:
def smart_text_to_sql_agent(user_question,conn,client,model):
  print("="*60)
  print(f"Question:{user_question}")
  print("="*60)

  schema_text = get_schema(conn)
  print("Generating SQL...")
  generated_sql = generate_sql(user_question, schema_text, client,model)
  print(f"SQL:{generate_sql}")

  result_df , error = execute_sql(generated_sql,conn)
  if error:
    print(f"Error executing SQL:{error}")
    return
  print(f"\nData ({len(result_df)} rows returned)")
  display(result_df)

  print("\nGenerating natural language answer...")
  answer = generate_answer(user_question,result_df, client, model)
  print("\n Answer:")
  print(answer)
  print("="*60)
smart_text_to_sql_agent("Who are the top 5 students in Programming",conn,client,model)

Question:Who are the top 5 students in Programming
Generating SQL...
SQL:<function generate_sql at 0x78f1e5277a60>

Data (5 rows returned)


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Nandita Rao
4,Arjun Nair



Generating natural language answer...

 Answer:
Based on the results, the top 5 students in Programming are: 

1. Aditya Kumar, 
2. Rohan Mehta, 
3. Anjali Singh, 
4. Nandita Rao, 
5. Arjun Nair.


In [44]:
smart_text_to_sql_agent(
    "Which subject has the highest average attendance?",
    conn, client, model
)

Question:Which subject has the highest average attendance?
Generating SQL...
SQL:<function generate_sql at 0x78f1e5277a60>

Data (1 rows returned)


,subject
0,Programming



Generating natural language answer...

 Answer:
Based on the results, it appears that the subject with the highest average attendance is actually not specified in the results, however, we do know that 'Programming' is a subject. Unfortunately, we do not have enough information to determine if 'Programming' has the highest average attendance.


## Practice Questions

### Beginner Questions

**Q1. What is the difference between a chatbot and an AI Agent? Give one example of each.**

- Chatbot: Answers user queries through conversation.
- AI Agent: Can reason, plan, and perform actions using tools.

Example:
- Chatbot: ChatGPT
- AI Agent: AutoGPT

---

**Q2. What is schema injection? Why does the AI need to know the schema before generating SQL?**

Schema injection means providing table and column information to the LLM.

The AI needs the schema to generate valid SQL queries using correct table and column names.

---

**Q3. Why do we set `temperature=0.0` when generating SQL queries?**

`temperature=0.0` makes the model deterministic and consistent.

This reduces random outputs and helps generate accurate SQL queries.

---

### Intermediate Questions

**Q4. In the `generate_sql()` function, what is the role of the `system` message vs the `user` message?**

- System Message: Gives instructions and rules to the AI.
- User Message: Contains the actual question to be converted into SQL.

---

**Q5. What does `PRAGMA table_info(students)` return? What information does it give us?**

It returns schema details of the `students` table such as:

- Column names
- Data types
- Primary key information
- Null constraints

---

**Q6. If the LLM generates incorrect SQL, what strategies can you use to improve it?**

- Provide a clearer schema.
- Use better prompt instructions.
- Add example SQL queries.
- Set temperature to 0.
- Validate generated SQL before execution.

## Coding Questions

 Q7. Modify `generate_sql()` to also handle aggregate functions like COUNT and SUM. Test it with the question: "How many students have an attendance above 90?"



In [45]:
def generate_sql(question, schema):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": f"""
                You are an SQL expert.
                Convert the user question into a valid SQLite query.
                The query may include aggregate functions such as COUNT, SUM, AVG, MIN, and MAX.

                Schema:
                {schema}

                Return only SQL.
                """
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content.strip()
question = "How many students have an attendance above 90?"
sql_query = generate_sql(question, schema)
print(sql_query)

SELECT COUNT(*) FROM students WHERE attendance > 90


Q8. Add a new function `get_distinct_values(column_name, conn)` that returns all unique values in a column. Use this to show the AI what values exist in the `subject` column.

In [46]:
def get_distinct_values(column_name, conn):
    cursor = conn.cursor()
    query = f"SELECT DISTINCT {column_name} FROM students"
    cursor.execute(query)

    return [row[0] for row in cursor.fetchall()]


subjects = get_distinct_values("subject", conn)

print("Available Subjects:")
for subject in subjects:
    print(subject)

Available Subjects:
Mathematics
Science
Programming


Q9. Build a simple loop that lets the user type questions and see results until they type 'exit'.

In [47]:
while True:
    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Exiting...")
        break

    sql_query = generate_sql(question, schema)

    print("\nGenerated SQL:")
    print(sql_query)

    try:
        result = execute_sql(sql_query, conn)

        print("\nResult:")
        for row in result:
            print(row)

    except Exception as e:
        print("Error:", e)


Ask a question (type 'exit' to quit): How many students are there?

Generated SQL:
SELECT COUNT(*) FROM students

Result:
   COUNT(*)
0        30
None

Ask a question (type 'exit' to quit): exit
Exiting...


## Mini Project: Natural Language SQL Dashboard

---

### Project Goal

Build a complete Natural Language to SQL system that:
1. Accepts a question from the user
2. Generates the SQL query
3. Executes it on the students database
4. Displays the results as a table
5. Shows a bar chart of the results (if numeric data is present)
6. Provides a natural language answer

In [48]:
pip install groq pandas matplotlib -q

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API")

conn = sqlite3.connect("students.db")

In [50]:
def get_schema(conn):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT sql
        FROM sqlite_master
        WHERE type='table'
    """)
    schema = ""
    for row in cursor.fetchall():
        schema += row[0] + "\n"
    return schema

In [51]:
def generate_sql(question, schema):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.0,
        messages=[
            {"role": "system","content": f"""
You are an SQLite expert.
Convert user questions into valid SQL.
Schema:
{schema}
Return only SQL.
"""
}, {
  "role": "user",
  "content": question
}])
    return response.choices[0].message.content.strip()

In [52]:
def execute_sql(query, conn):
    df = pd.read_sql_query(query, conn)
    return df

In [53]:
def generate_answer(question, results):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.3,
        messages=[
            {
                "role": "system",
                "content": "Answer the user's question using the SQL results."
            },
            {"role": "user","content": f"""
Question:
{question}
Results:
{results.to_string(index=False)}
""" } ] )
    return response.choices[0].message.content

In [54]:
def plot_chart(df):
    numeric_cols = df.select_dtypes(include="number").columns
    if len(numeric_cols) == 0:
        print("No numeric data available for chart.")
        return
    if len(df.columns) < 2:
        print("Not enough columns to plot.")
        return
    x_col = df.columns[0]
    y_col = numeric_cols[0]
    plt.figure(figsize=(8,5))
    plt.bar(df[x_col].astype(str), df[y_col])
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title("Query Results")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [55]:
schema = get_schema(conn)
while True:
    question = input("\nAsk a question (type 'exit' to quit): ")
    if question.lower() == "exit":
        break
    sql_query = generate_sql(question, schema)
    print("\nGenerated SQL:")
    print(sql_query)
    try:
        results = execute_sql(sql_query, conn)
        print("\nResults:")
        print(results)
        print("\nNatural Language Answer:")
        answer = generate_answer(question, results)
        print(answer)
        plot_chart(results)
    except Exception as e:
        print("Error:", e)


Ask a question (type 'exit' to quit): How many students are there?

Generated SQL:
SELECT COUNT(*) FROM students;
Error: Execution failed on sql 'SELECT COUNT(*) FROM students;': no such table: students

Ask a question (type 'exit' to quit): Which student has the highest attendance?

Generated SQL:
```sql
SELECT s.student_name, s.student_id, COUNT(a.attendance_id) as total_attendance
FROM students s
JOIN attendance a ON s.student_id = a.student_id
GROUP BY s.student_id, s.student_name
ORDER BY total_attendance DESC
LIMIT 1;
```

However, if you want to get the student with the highest attendance without considering ties, you can use the following query:

```sql
SELECT s.student_name, s.student_id, COUNT(a.attendance_id) as total_attendance
FROM students s
JOIN attendance a ON s.student_id = a.student_id
GROUP BY s.student_id, s.student_name
ORDER BY total_attendance DESC
LIMIT 1;
```

If you want to get all students with the highest attendance, you can use the following query:

```sql

--END OF MINIPROJECT DAY-9--